# 06 · Validate irrigation technology evidence

Checks the approved 824 country-year technology shares and surface, sprinkler and drip delivery efficiencies.

This notebook contains the scientific Python logic directly, not only a call to a .py script. Run cells in order after placing approved inputs under `data/`. All outputs are written to new directories under `runs/`; source files are never overwritten. For scientific scope and frozen evidence dependencies, see `docs/reproducibility.md`.

## Scientific methods and checks

In [ ]:
#!/usr/bin/env python3
"""Check approved country-year technology shares and delivery efficiencies."""
from __future__ import annotations
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd

SHARES = ['surface_share','sprinkler_share','drip_share']
EFFICIENCIES = {'surface':0.42,'sprinkler':0.7125,'drip':0.855}


def check_shares(df, expected_rows=824):
    keys=['country_iso3','year']
    missing=set(keys+SHARES)-set(df.columns)
    if missing:raise ValueError(f'Missing share fields: {sorted(missing)}')
    if df[keys].isna().any().any() or df.duplicated(keys).any():raise ValueError('Duplicate/null country-year technology keys')
    values=df[SHARES].apply(pd.to_numeric,errors='raise').to_numpy(float)
    if not np.isfinite(values).all() or np.any(values < 0) or np.any(values > 1):
        raise ValueError('Nonfinite or out-of-range technology shares')
    if not np.allclose(values.sum(axis=1),1.0,atol=5e-3,rtol=0):
        raise ValueError('Some approved technology shares fail sum-to-one tolerance 0.005')
    if expected_rows is not None and len(df)!=expected_rows:raise ValueError(f'Expected {expected_rows} technology rows, got {len(df)}')
    return {'rows':len(df),'countries':int(df.country_iso3.nunique()),'duplicate_keys':0,'share_sum_status':'PASS'}


def efficiency_values(df):
    req=['technology','final_delivery_efficiency']
    missing=set(req)-set(df.columns)
    if missing:raise ValueError(f'Missing efficiency fields: {sorted(missing)}')
    values=df.set_index('technology')['final_delivery_efficiency']
    if values.index.duplicated().any():raise ValueError('Duplicate efficiency systems')
    for system,expected in EFFICIENCIES.items():
        if system not in values.index or not np.isclose(float(values[system]), expected, atol=1e-12,rtol=0):
            raise ValueError(f'Unexpected {system} efficiency; scientific re-approval needed')
    return {system:float(values[system]) for system in EFFICIENCIES}


## Inputs and output location

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'
SHARES_FILE = DATA / 'technology' / '12_FABLE_irrigation_technology_shares_2000_2020_FINAL.csv'
EFFICIENCY_FILE = DATA / 'technology' / '13_FABLE_irrigation_efficiency_parameters_FINAL.csv'


## Execute and inspect

In [ ]:
for path in (SHARES_FILE, EFFICIENCY_FILE):
    if not path.is_file():
        raise FileNotFoundError(path)
share_report = check_shares(pd.read_csv(SHARES_FILE), expected_rows=824)
share_report['efficiencies'] = efficiency_values(pd.read_csv(EFFICIENCY_FILE))
print(json.dumps(share_report, indent=2))
print('STATUS: PASS_APPROVED_TECHNOLOGY_EVIDENCE_QC')
